In [ ]:
'''import zipfile

zip_path = "/content/drive/MyDrive/whu-building-dataset.zip"
extract_path = "/content/WHU/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully")'''

Unzipped successfully


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [ ]:
!pip install tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [ ]:
class WHUDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, 0)

        image = cv2.resize(image, (256,256))
        mask = cv2.resize(mask, (256,256))

        image = image / 255.0
        mask = mask / 255.0

        image = torch.tensor(image).permute(2,0,1).float()
        mask = torch.tensor(mask).unsqueeze(0).float()

        return image, mask

In [ ]:
train_dataset = WHUDataset(
    r"C:\anilminiproject\datasets\WHU\train\Image",
    r"C:\anilminiproject\datasets\WHU\train\Mask"
)
'''train_dataset = WHUDataset(
    "/content/WHU/WHU/train/Image",
    "/content/WHU/WHU/train/Mask"
)'''

val_dataset = WHUDataset(
    r"C:\anilminiproject\datasets\WHU\val\Image",
    r"C:\anilminiproject\datasets\WHU\val\Mask"
)
'''val_dataset = WHUDataset(
    "/content/WHU/WHU/val/Image",
    "/content/WHU/WHU/val/Mask"
)'''
test_dataset = WHUDataset(
    r"C:\anilminiproject\datasets\WHU\test\Image",
    r"C:\anilminiproject\datasets\WHU\test\Mask"
)
'''test_dataset = WHUDataset(
    "/content/WHU/WHU/test/Image",
    "/content/WHU/WHU/test/Mask"
)'''

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, in_channels):
        super(AttentionBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, in_channels//2, 1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels//2, in_channels, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        att = self.conv1(x)
        att = self.relu(att)
        att = self.conv2(att)
        att = self.sigmoid(att)
        return x * att

In [ ]:
class DSATNet(nn.Module):
    def __init__(self):
        super(DSATNet, self).__init__()

        # Encoder Block 1
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)

        # Encoder Block 2
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)

        # Dual-scale branch
        self.dual_conv = nn.Conv2d(128, 128, 5, padding=2)

        # Attention
        self.att = AttentionBlock(128)

        # Decoder
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )

        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        self.final = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x1 = self.enc1(x)
        p1 = self.pool1(x1)

        x2 = self.enc2(p1)
        p2 = self.pool2(x2)

        dual = self.dual_conv(p2)
        att = self.att(dual)

        up1 = self.up1(att)
        d1 = self.dec1(up1)

        up2 = self.up2(d1)
        d2 = self.dec2(up2)

        out = self.final(d2)
        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = DSATNet().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

cuda


In [ ]:
def calculate_metrics(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()

    pred = pred.view(-1)
    target = target.view(-1)

    TP = (pred * target).sum().item()
    TN = ((1 - pred) * (1 - target)).sum().item()
    FP = (pred * (1 - target)).sum().item()
    FN = ((1 - pred) * target).sum().item()

    epsilon = 1e-7

    accuracy = (TP + TN) / (TP + TN + FP + FN + epsilon)
    precision = TP / (TP + FP + epsilon)
    recall = TP / (TP + FN + epsilon)
    f1 = 2 * precision * recall / (precision + recall + epsilon)
    iou = TP / (TP + FP + FN + epsilon)
    dice = (2 * TP) / (2 * TP + FP + FN + epsilon)

    return accuracy, precision, recall, f1, iou, dice


In [ ]:
epochs = 25
for epoch in range(epochs):

    model.train()
    train_loss = 0

    for images, masks in tqdm(train_loader):

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} "
          f"Val Loss: {val_loss/len(val_loader):.4f}")

100%|████████████████████████████████████████████████████████████████████████████████| 717/717 [02:54<00:00,  4.10it/s]


Epoch [1/25] Train Loss: 0.2267 Val Loss: 0.1505


100%|████████████████████████████████████████████████████████████████████████████████| 717/717 [02:44<00:00,  4.36it/s]


Epoch [2/25] Train Loss: 0.1362 Val Loss: 0.1137


100%|████████████████████████████████████████████████████████████████████████████████| 717/717 [02:05<00:00,  5.70it/s]


...

Epoch [23/25] Train Loss: 0.0661 Val Loss: 0.0664


100%|████████████████████████████████████████████████████████████████████████████████| 717/717 [02:05<00:00,  5.70it/s]


Epoch [24/25] Train Loss: 0.0655 Val Loss: 0.0687


100%|████████████████████████████████████████████████████████████████████████████████| 717/717 [02:05<00:00,  5.69it/s]


Epoch [25/25] Train Loss: 0.0643 Val Loss: 0.0641


In [ ]:
torch.save(model.state_dict(),
           r"C:\anilminiproject\trainedmodels\dsatnet_whu.pth")

In [ ]:
    model.eval()
    val_loss = 0
    total_metrics = np.zeros(6)

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()

            metrics = calculate_metrics(outputs, masks)
            total_metrics += np.array(metrics)

    avg_metrics = total_metrics / len(val_loader)

    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}")
    print(f"Val Loss: {val_loss/len(val_loader):.4f}")
    print(f"Accuracy: {avg_metrics[0]:.4f}")
    print(f"Precision: {avg_metrics[1]:.4f}")
    print(f"Recall: {avg_metrics[2]:.4f}")
    print(f"F1 Score: {avg_metrics[3]:.4f}")
    print(f"IoU: {avg_metrics[4]:.4f}")
    print(f"Dice: {avg_metrics[5]:.4f}")


Epoch [3/3]
Train Loss: 0.1214
Val Loss: 0.1058
Accuracy: 0.9584
Precision: 0.8683
Recall: 0.8475
F1 Score: 0.8564
IoU: 0.7505
Dice: 0.8564
